In [4]:
import pandas as pd
import urllib.parse
import urllib.error
import time

def get_atn_inventory(search_terms):
    print("Frage ATN ERDDAP Server nach Datensaetzen ab...")
    
    # Korrigierte Basis-URL
    base_url = "https://atn.ioos.us/erddap/search/index.csv"
    
    all_datasets = []
    
    for term in search_terms:
        print(f"Suche nach: {term}")
        
        query_url = f"{base_url}?page=1&itemsPerPage=1000&searchFor={urllib.parse.quote(term)}"
        
        try:
            # WICHTIG: ERDDAP liefert in Zeile 2 die Datentypen, die wir ueberspringen muessen
            df = pd.read_csv(query_url, skiprows=[1])
            
            if 'tabledap' in df.columns:
                df = df.dropna(subset=['tabledap'])
                df['search_term'] = term
                all_datasets.append(df)
                
        except urllib.error.HTTPError as e:
            if e.code == 404:
                print(f"-> Keine Treffer fuer '{term}'.")
            else:
                print(f"-> HTTP Fehler bei {term}: {e}")
        except Exception as e:
            print(f"-> Fehler bei {term}: {e}")
            
        # Kurze Pause schont den Server
        time.sleep(1)
        
    if not all_datasets:
        print("Keine Datensaetze gefunden.")
        return pd.DataFrame()
        
    inventory_df = pd.concat(all_datasets, ignore_index=True)
    inventory_df = inventory_df.drop_duplicates(subset=['Dataset ID'])
    
    return inventory_df

if __name__ == "__main__":
    terms = ["Cetacea", "Phocidae", "Cheloniidae", "Whale", "Seal", "Turtle"]
    
    inventory = get_atn_inventory(terms)
    
    if not inventory.empty:
        print(f"\nErfolg: {len(inventory)} Tracking-Datensaetze im ATN gefunden.")
        
        clean_inventory = inventory[['Dataset ID', 'Title', 'Institution', 'Summary', 'search_term']]
        clean_inventory.to_csv("atn_finflow_inventory.csv", index=False)
        print("Inventar wurde als 'atn_finflow_inventory.csv' gespeichert.\n")
        
        print("Beispiel-Datensaetze:")
        for _, row in clean_inventory.head(3).iterrows():
            print(f"- ID: {row['Dataset ID']}")
            print(f"  Titel: {row['Title'][:80]}...")
            print(f"  Institution: {row['Institution']}")
            print()
    else:
        print("Es konnten keine relevanten Datensaetze erstellt werden.")

Frage ATN ERDDAP Server nach Datensaetzen ab...
Suche nach: Cetacea
-> Keine Treffer fuer 'Cetacea'.
Suche nach: Phocidae
-> Keine Treffer fuer 'Phocidae'.
Suche nach: Cheloniidae
-> Keine Treffer fuer 'Cheloniidae'.
Suche nach: Whale
-> Keine Treffer fuer 'Whale'.
Suche nach: Seal
-> Keine Treffer fuer 'Seal'.
Suche nach: Turtle
-> Keine Treffer fuer 'Turtle'.
Keine Datensaetze gefunden.
Es konnten keine relevanten Datensaetze erstellt werden.
